In [ ]:
!pip install sentence-transformers


In [10]:
!pip install faiss-cpu

Defaulting to user installation because normal site-packages is not writeable


In [1]:
# Fetch and Process the Text
import re
from sklearn.datasets import fetch_20newsgroups
from pprint import pprint 
print("Library Loaded Succesfully")


Library Loaded Succesfully


In [2]:
# Fetch the raw dataset
# Raw full data set without any skipings
print("Downloading...")
newsgroups_raw = fetch_20newsgroups(subset='train', remove=())

print(f"Total documents loaded : {len(newsgroups_raw.data)}")
print(f"Catogories : {len(newsgroups_raw.target_names)}")
pprint(list(newsgroups_raw.target_names))

# Looking How Raw Dataset
print("\n--- Raw Document Sample ---")
print(newsgroups_raw.data[0][:500]) # first 500 characters

Downloading...
Total documents loaded : 11314
Catogories : 20
['alt.atheism',
 'comp.graphics',
 'comp.os.ms-windows.misc',
 'comp.sys.ibm.pc.hardware',
 'comp.sys.mac.hardware',
 'comp.windows.x',
 'misc.forsale',
 'rec.autos',
 'rec.motorcycles',
 'rec.sport.baseball',
 'rec.sport.hockey',
 'sci.crypt',
 'sci.electronics',
 'sci.med',
 'sci.space',
 'soc.religion.christian',
 'talk.politics.guns',
 'talk.politics.mideast',
 'talk.politics.misc',
 'talk.religion.misc']

--- Raw Document Sample ---
From: lerxst@wam.umd.edu (where's my thing)
Subject: WHAT car is this!?
Nntp-Posting-Host: rac3.wam.umd.edu
Organization: University of Maryland, College Park
Lines: 15

 I was wondering if anyone out there could enlighten me on this car I saw
the other day. It was a 2-door sports car, looked to be from the late 60s/
early 70s. It was called a Bricklin. The doors were really small. In addition,
the front bumper was separate from the rest of the body. This is 
all I know. If anyone can tellme

In [3]:
# custom Cleaning Engine
def clean_document(text):
    """ Cleans a 20 Newsgroups document for semantic embedding.
    
    JUSTIFICATION FOR CHOICES:
    1. KEEP the Subject line: The subject is essentially a dense summary of the document's 
       semantic intent. Discarding it loses high-value clustering signal.
    2. DISCARD standard headers (From, Organization, NNTP-Posting-Host): We want to cluster 
       based on semantic topics, not group documents by the same author or university network.
    3. DISCARD quoted text (lines starting with >): Quoted text represents previous emails 
       in a thread. Keeping it causes the model to over-index on thread continuity rather 
       than the specific content of the new reply.
    4. DISCARD signature blocks: Often separated by '--', these contain quotes, ASCII art, 
       or contact info that act as noise in vector space.
    """

    # 1 Extarct the Subject line
    subject_match = re.search(r'^Subject:\s*(.*)' ,text, flags = re.MULTILINE | re.IGNORECASE)
    subject = subject_match.group(1).strip() if subject_match else ""

    # 2 Stipe all headers and standard email format
    parts = text.split('\n\n', 1)
    body = parts[1] if len(parts) > 1 else text

    # 3 Remove Quoted lines (lines starts with > , |, or in the article..)
    body_lines = body.split('\n')
    unquoted_lines = [
        line for line in body_lines
        if not re.match(r'^[\s]*[>\|]', line)
        and not line.lower().startswith('in article')
    ]

    # Remove signature  blocks( everything after standard '--' separators)
    clean_body_text = []
    for line in unquoted_lines:
        if line.strip() == '--':
            break # stops when signature blocks hit
        clean_body_text.append(line)

    cleaned_body = ' '.join(clean_body_text)

    # Clean up extra whitespace and new lines
    cleaned_body = re.sub(r'\s+',' ',cleaned_body).strip()

    # Combine the high- Signal with the cleanned body
    final_text = f"{subject}.{cleaned_body}".strip()
    return final_text
print("Cleaning Function Defined...")

        
    

Cleaning Function Defined...


In [4]:
# Apply the cleaning function to all documents
cleaned_docs = [clean_document(doc) for doc in newsgroups_raw.data]

# Filter out documents that became empty after cleaning (e.g., posts that were 100% quotes)
valid_indices = [i for i, doc in enumerate(cleaned_docs) if len(doc) > 20]
cleaned_docs = [cleaned_docs[i] for i in valid_indices]
labels = [newsgroups_raw.target[i] for i in valid_indices]

print(f"Documents remaining after dropping empty/junk posts: {len(cleaned_docs)}")

print("\n--- CLEANED DOCUMENT SAMPLE ---")
print(cleaned_docs[0][:500])

Documents remaining after dropping empty/junk posts: 11299

--- CLEANED DOCUMENT SAMPLE ---
WHAT car is this!?.I was wondering if anyone out there could enlighten me on this car I saw the other day. It was a 2-door sports car, looked to be from the late 60s/ early 70s. It was called a Bricklin. The doors were really small. In addition, the front bumper was separate from the rest of the body. This is all I know. If anyone can tellme a model name, engine specs, years of production, where this car is made, history, or whatever info you have on this funky looking car, please e-mail. Thanks


In [5]:
# Generate the Sementic Embaddings with "all-MiniLM-L6-v2"
from sentence_transformers import SentenceTransformer
import numpy as np
import time


In [7]:
# Load the embedding model
# JUSTIFICATION: all-MiniLM-L6-v2 is chosen because it is highly optimized for semantic search.
# It produces 384-dimensional vectors, striking an ideal balance between capturing deep 
# semantic meaning and maintaining the lightweight performance required for this system.

print("Loading Embedding model (all-MiniLM-L6-v2)..")
model = SentenceTransformer('all-MiniLM-L6-v2')

print(f"Encoding {len(cleaned_docs)} documents. This may take Time ")
start_time = time.time()

# Encode the cleaned Documents into vectors
# Visual indicator in Jupyter

embeddings  = model.encode(cleaned_docs,show_progress_bar=True)

end_time = time.time()
print(f"Encoding complete in {end_time -start_time:.2f} seconds.")
print(f"Embedding Shape: {embeddings.shape} (Documet xx Dimentions)")




Loading Embedding model (all-MiniLM-L6-v2)..


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Encoding 11299 documents. This may take Time 


Batches:   0%|          | 0/354 [00:00<?, ?it/s]

Encoding complete in 560.02 seconds.
Embedding Shape: (11299, 384) (Documet xx Dimentions)


In [14]:
# Building and Persisting vector database
import faiss
import pickle 
import os

In [17]:
# craete a directory to store our persisted data
os.makedirs("data",exist_ok =True)

In [23]:
# 1. Initialize the FAISS Index
# JUSTIFICATION: FAISS (Facebook AI Similarity Search) is chosen over a standalone vector database 
# (like Pinecone or Weaviate) because it operates entirely in-memory as a lightweight C++ library. 
# We use IndexFlatL2 for exact nearest neighbor search, which is highly efficient for a corpus 
# of ~20,000 documents without the overhead of network calls.

dimention = embeddings.shape[1]
index = faiss.IndexFlatL2(dimention)

print(f"FAISS index initialized. Is index trained ? {index.is_trained}")

# 2 Add vectors to the index
index.add(np.array(embeddings))
print(f"Total vectors stored in FAISS : {index.ntotal}")

# 3 Persist the index to the disk
faiss.write_index(index, "data/corpus_index.faiss")
print("FAISS index saved to 'data/corpus_index.faiss'")

# 4  Persist the actual text and labels 

corpus_metadata = {
    "documents": cleaned_docs,
    "labels": labels
}
with open("data/corpus_metadata.pkl", "wb") as f:
    pickle.dump(corpus_metadata, f)
    print("Corpus metadata saved to 'datat/corpus_metadata.pkl'")

FAISS index initialized. Is index trained ? True
Total vectors stored in FAISS : 11299
FAISS index saved to 'data/corpus_index.faiss'
Corpus metadata saved to 'datat/corpus_metadata.pkl'


In [32]:
from sklearn.mixture import GaussianMixture
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

In [ ]:
# We will test a range of cluster counts to find the mathematical "elbow" or minimum.
# NOTE: Running GMM on 20,000 dense vectors can be slow. If this takes too long, 
# you can test on a random subset of `embeddings` (e.g., 5000 samples) just for this BIC step.

print("Applying PCA to reduce the dimentions before clusuturing...")
# Reduce to 50 diemntions 
pca =PCA(n_components = 50, random_state =42)
reduced_embeddings = pca.fit_transform(embeddings)


# Let's expand our search range 
n_components_range  = range(10,42,2)
bic_scores = []
print("Calculating BIC scores to justify the number of clusters....")
for n in n_components_range:
    # covariance_type='spherical' is faster and often works well for high-dimensional text embeddings

    gmm = GaussianMixture(n_components = n, covariance_type ='spherical', random_state = 42)
    gmm.fit(embeddings)
    bic_scores.append(gmm.bic(embeddings))
    print(f"Tested {n} clusters: BIC = {bic_scores[-1]:.0f}")

# plot the Evidence 
plt.figure(figsize=(10,6))
plt.plot(n_components_range, bic_scores, marker='o')
plt.title('BIC Scores per Number of Clusters (Lower is Better)')
plt.xlabel('Number of Clusters')
plt.ylabel('BIC Score')
plt.grid(True)
plt.show()
    

In [40]:
# Find Optimal Clusters automatically

def find_optimal_cluster(x_values, y_values):
     # Automatically finds the 'elbow' in a curve using the point-to-line distance method.
    x = np.array(x_values)
    y = np.array(y_values)

    # Get Coordinates of the first and last points
    p1 = np.array([x[0],y[0]])
    p2 = np.array([x[-1],y[-1]])

    distances = []
    for i in range(len(x)):
        p3 = np.array([x[i], y[i]])
        
        # Vector 1 (from p1 to p2) and Vector 2 (from p3 to p1)
        v1 = p2 - p1
        v2 = p1 - p3
        
        # Calculate the 2D cross product magnitude manually: (x1 * y2) - (y1 * x2)
        cross_2d = (v1[0] * v2[1]) - (v1[1] * v2[0])
        
        # Calculate the perpendicular distance
        distance = np.abs(cross_2d) / np.linalg.norm(v1)
        distances.append(distance)

    # Optimal Cluster count
    optimal_index = np.argmax(distances)
    return x[optimal_index]

# calulates cluster count automatically
OPTIMAL_CLUSTERS = find_optimal_cluster(list(n_components_range),bic_scores)
        
print(f"--- AUTOMATED ELBOW DETECTION ---")
print(f"Mathematically optimal number of clusters detected: {OPTIMAL_CLUSTERS}")
        
        

--- AUTOMATED ELBOW DETECTION ---
Mathematically optimal number of clusters detected: 22


In [82]:
class SemanticCache:
    def __init__(self, embedding_model, gmm_model,threshold=0.85):
        """
        Initializes the Semantic Cache.
        
        JUSTIFICATION FOR TUNABLE DECISION (Threshold):
        The 'threshold' (default 0.85) dictates the strictness of the cache. 
        - A high value (e.g., 0.95) acts almost like an exact-match cache: high accuracy, 
          but very low hit rate. 
        - A low value (e.g., 0.70) yields a high hit rate, but risks returning an answer 
          about "baseball" when the user asked about "hockey" because they share a semantic 
          cluster. 0.85 offers a balanced sweet spot for conversational NLP."""

        self.embedding_model =embedding_model
        self.gmm_model = gmm_model
        self.threshold = threshold

        # The core data Structure : A dictionary partitioned cluster ID
        self.cache_store = {} 

        # State management for API
        self.total_entries = 0;
        self.hit_count = 0;
        self.miss_count =0;
        
    def _cosine_similarity(self,vec1, vec2):
        # calculates cosine similarity between 2 vectors 
        dot_product =np.dot(vec1,vec2)
        norm_a = np.linalg.norm(vec1)
        norm_b = np.linalg.norm(vec2)
        if norm_a == 0 or norm_b ==0:
            return 0.0
        return dot_product / (norm_a * norm_b)

    def check_cache(self,query: str):
        """
        Checks if a semantically similar query exists in the cache.
        Returns (is_hit, matched_query, score, cached_result, dominant_cluster).
        """

        # 1 Embed the incomming query
        query_embedding  = self.embedding_model.encode([query])[0]

        # 2 Find the dominent cluster using GMM
        cluster_probs  = self.gmm_model.predict_proba([query_embedding])[0]
        dominant_cluster = int(np.argmax(cluster_probs))

        # 3 Cluster bin doesn't exits it's automatic miss
        if dominant_cluster not in self.cache_store or not self.cache_store[dominant_cluster]:
            return False, None, 0.0, None, dominant_cluster

        # 4. Search ONLY within the dominant cluster
        best_score = -1.0
        best_match = None

        for cached_item in self.cache_store[dominant_cluster]:
            score =self._cosine_similarity(query_embedding, cached_item['embedding'])
            if score > best_score:
                best_score = score
                best_match= cached_item
        # 5. Evaluate against our tunable threshold

        if best_score >= self.threshold:
            self.hit_count +=1
            return True, best_match['query'], float(best_score), best_match['result'], dominant_cluster

        return False, None,0.0, None, dominant_cluster

    def add_to_cache(self, query: str, query_embedding, result: str, dominant_cluster: int):
        """Stores a new query and its result in the appropriate cluster bin."""
        if dominant_cluster not in self.cache_store:
            self.cache_store[dominant_cluster] = []
            
        self.cache_store[dominant_cluster].append({
            'query': query,
            'embedding': query_embedding,
            'result': result
        })
        
        self.total_entries += 1
        self.miss_count += 1 # If we are adding it, it means it was a miss

    def get_stats(self):
        """Returns the current state of the cache for the API."""
        total_requests = self.hit_count + self.miss_count
        hit_rate = (self.hit_count / total_requests) if total_requests > 0 else 0.0
        
        return {
            "total_entries": self.total_entries,
            "hit_count": self.hit_count,
            "miss_count": self.miss_count,
            "hit_rate": round(hit_rate, 3)
        }
        
    def flush(self):
        """Clears the cache entirely."""
        self.cache_store = {}
        self.total_entries = 0
        self.hit_count = 0
        self.miss_count = 0

In [83]:
from sklearn.mixture import GaussianMixture
import joblib

# 1. Lock in our mathematically justified cluster count
OPTIMAL_CLUSTERS = 22 

print(f"Training final GMM with {OPTIMAL_CLUSTERS} clusters...")
final_gmm = GaussianMixture(n_components=OPTIMAL_CLUSTERS, covariance_type='spherical', random_state=42)

# Train the model using the embeddings we created earlier
# (Make sure you have already run the cell that created the 'embeddings' variable!)
final_gmm.fit(embeddings)
print("GMM trained successfully!")

# Let's save it to disk just to be safe for Part 4
joblib.dump(final_gmm, 'data/gmm_model.pkl')

# 2. Now, Initialize the Semantic Cache
# We use a threshold of 0.85 as our tunable decision for semantic strictness
semantic_cache = SemanticCache(embedding_model=model, gmm_model=final_gmm, threshold=0.85)

print("Semantic Cache initialized successfully in memory!")

Training final GMM with 22 clusters...
GMM trained successfully!
Semantic Cache initialized successfully in memory!


In [87]:
# 1. Initialize the Semantic Cache
# We use a threshold of 0.85 as our tunable decision for semantic strictness
semantic_cache = SemanticCache(embedding_model=model, gmm_model=final_gmm, threshold=0.75)
print("Semantic Cache initialized successfully in memory!")

Semantic Cache initialized successfully in memory!


In [88]:
import numpy as np

def test_query(user_query):
    print(f"\n--- Processing Query: '{user_query}' ---")
    
    # 1. Check the semantic cache
    is_hit, matched_query, score, cached_result, dominant_cluster = semantic_cache.check_cache(user_query)
    
    if is_hit:
        print("🟢 CACHE HIT!")
        print(f"Matched with previous query: '{matched_query}'")
        print(f"Similarity Score: {score:.3f}")
        print(f"Dominant Cluster: {dominant_cluster}")
        return
        
    print("🔴 CACHE MISS! Computing result from scratch...")
    
    # 2. On a miss, embed the query and search the FAISS vector database
    query_embedding = model.encode([user_query])[0]
    
    # FAISS requires a 2D array, so we reshape it: np.array([query_embedding])
    distances, indices = index.search(np.array([query_embedding]), k=1)
    
    best_match_idx = indices[0][0]
    real_result = cleaned_docs[best_match_idx]
    
    # 3. Store the newly computed result in the cache [cite: 52]
    # We ask the GMM for the dominant cluster again just to log it properly
    cluster_probs = final_gmm.predict_proba([query_embedding])[0] 
    dominant_cluster = int(np.argmax(cluster_probs))
    
    semantic_cache.add_to_cache(user_query, query_embedding, real_result, dominant_cluster)
    
    print(f"Result found and cached. Dominant Cluster: {dominant_cluster}")
    print(f"Preview of document: {real_result[:100]}...")

In [89]:
# Test 1: A brand new question (Should be a MISS)
test_query("What is the best orbit for a space satellite?")

# Test 2: The exact same semantic intent, but phrased differently (Should be a HIT)
test_query("Which orbital path is optimal for satellites in space?")

# Test 3: A completely different topic (Should be a MISS)
test_query("How do I encrypt a hard drive using strong cryptography?")

# Test 4: Verify the stats endpoint logic [cite: 53, 54]
print("\n--- Current Cache Stats ---")
print(semantic_cache.get_stats())


--- Processing Query: 'What is the best orbit for a space satellite?' ---
🔴 CACHE MISS! Computing result from scratch...
Result found and cached. Dominant Cluster: 5
Preview of document: Re: Orbital RepairStation.But with cheaper fuel from space based sources it will be cheaper to reach...

--- Processing Query: 'Which orbital path is optimal for satellites in space?' ---
🟢 CACHE HIT!
Matched with previous query: 'What is the best orbit for a space satellite?'
Similarity Score: 0.813
Dominant Cluster: 5

--- Processing Query: 'How do I encrypt a hard drive using strong cryptography?' ---
🔴 CACHE MISS! Computing result from scratch...
Result found and cached. Dominant Cluster: 5
Preview of document: Re: How to the disks copy protected..Uh oh... Umm, there are a number of copy protection schemes. So...

--- Current Cache Stats ---
{'total_entries': 2, 'hit_count': 1, 'miss_count': 2, 'hit_rate': 0.333}
